# Wound type + T / I / M baseline (Colab)

Trains the multi-task baseline on the **Wound categorisation** folder (TYPE 1 to TYPE 8) directly from Google Drive.
T, I and M labels are derived from the type, following the TIME scoring in the clinical manual (type = T + I + M).

**Before running:** `Runtime > Change runtime type > GPU`. The folder is shared with you, so add a shortcut to it in My Drive
(right-click the folder in Drive > *Organise* > *Add shortcut* > My Drive), or edit `DRIVE_DATA_DIR` below.
Then `Runtime > Run all`. The photos stay in your Drive and this Colab session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA_DIR = '/content/drive/MyDrive/Wound categorisation'   # edit if your shortcut lives elsewhere
DRIVE_OUT_DIR = '/content/drive/MyDrive/wound_baseline_runs'      # results are copied here at the end
BRANCH = 'claude/project-thread-xhzi9m'

In [ ]:
# Copy the images to local disk once: training reads every image each epoch, and Drive I/O is slow.
!rm -rf /content/wounds && mkdir -p /content/wounds
!cp -r "$DRIVE_DATA_DIR"/TYPE* /content/wounds/
!ls /content/wounds && find /content/wounds -type f | wc -l

In [ ]:
!rm -rf /content/repo && git clone -q --depth 1 -b $BRANCH https://github.com/nuradib-pn/nuradibmaspo-hjyusof.git /content/repo
%cd /content/repo/wound_classifier
!pip -q install -r requirements.txt

## 1. Survey (class counts, image sizes, exact duplicates)

In [ ]:
!python -m wound_tim.survey --data-root /content/wounds --out /content/survey.json | head -40

## 2. Train

ResNet-50 pretrained on ImageNet, four heads (8-way type, T, I, M), split 70 / 15 / 15 stratified by type and grouped by
capture date so same-day photos never straddle train and test. About 10 to 15 minutes on a T4 GPU.

In [ ]:
!python -m wound_tim.train --config configs/baseline.yaml --data-root /content/wounds --out-dir /content/runs/resnet50

## 3. Results on the held-out test split

In [ ]:
import json, pandas as pd
r = json.load(open('/content/runs/resnet50/metrics.json'))
print('split sizes:', r['split_sizes'], '| capture-day groups:', r['n_groups'], '| best epoch:', r['best_epoch'])
pd.DataFrame({h: {k: round(v, 3) for k, v in m.items() if k in ('n', 'accuracy', 'balanced_accuracy', 'macro_f1')}
              for h, m in r['test'].items()}).T

In [ ]:
import matplotlib.pyplot as plt, numpy as np
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, head in zip(axes, ['wound_type', 'wound_type_from_tim']):
    cm = np.array(r['test'][head]['confusion_matrix'])
    ax.imshow(cm, cmap='Blues')
    labels = [v.split('-')[0].strip() for v in r['vocabs'][head]]
    ax.set_xticks(range(len(labels)), labels, rotation=45); ax.set_yticks(range(len(labels)), labels)
    for (i, j), v in np.ndenumerate(cm): ax.text(j, i, v, ha='center', va='center', fontsize=8)
    ax.set_title(head.replace('_', ' ')); ax.set_xlabel('predicted'); ax.set_ylabel('true')
plt.tight_layout()

In [ ]:
!mkdir -p "$DRIVE_OUT_DIR" && cp -r /content/runs/resnet50 "$DRIVE_OUT_DIR"/ && cp /content/survey.json "$DRIVE_OUT_DIR"/
print('saved to', DRIVE_OUT_DIR)